# 贝叶斯后验：Poisson-Gamma 共轭模型

> 对应代码：`MathStatsCode/code_in_notes/Chap.16/bayes_lambda_gamma.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：在 (α,β) 网格上计算 Poisson 似然与 Gamma 先验下的后验（重参数化 α=a²/b, β=b/a），并绘制后验等高线图。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.16`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.16")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

`clear all` 清空数据。本 notebook 用网格法计算 **Poisson-Gamma 共轭模型**的后验分布：先验 λ~Gamma(5,2)，观测 X|λ~Poisson(λ)。


In [ ]:
%%stata
clear all


**generate data, P(10)**

生成数据并汇总充分统计量：`set obs 100`、`set seed 15634145`；`gen lambda=rgamma(5,2)`（每个观测的先验“真值”）、`gen x=rpoisson(lambda)`（观测值）；`sum x` 得到 sum_x=Σx、N=100——Poisson 似然只通过这两个量依赖数据。


In [ ]:
%%stata
set obs 100
set seed 15634145
gen lambda = rgamma(5,2)
gen x = rpoisson(lambda)
sum x
local sum_x = r(sum)
local N = _N


**compute posterior**

在 (a,b) 网格上计算**重参数化后的后验**（α=a²/b、β=b/a，使 α 与 β 的取值更均匀），最后画后验等高线图：
- `frame create posterior ...` 新建结果框；双重 `forvalues a=5(0.03)15`、`forvalues b=1(0.05)40` 遍历网格；
- 每个 (a,b) 点：先由 a、b 换算 α、β，计算对数后验 `sum_x·log(β)+ΣlnΓ(α+x)-N·lnΓ(α)-(α·N+sum_x)·log(1+β)`（去掉与参数无关的常数项）；
- `frame post` 把 (a,b,α,β,后验值) 写入结果框，`di "." _continue` 打印进度点；
- 循环结束后，`frame posterior:` 前缀可以在另一个数据框上直接画图：`contourline` 等高线 + 标出后验最高点的 `scatter/mlabel`；`graph export` 导出。
网格法（grid approximation）是理解贝叶斯后验最直观的方法：把参数空间离散化、逐点计算非归一化的后验密度。


In [ ]:
%%stata
frame create posterior a b alpha beta posterior
forvalues a=5(0.03)15{
	forvalues b=1(0.05)40{
		local alpha = `a'^2/`b'
		local beta = `b'/`a'
		gen log_gamma_alpha_p_x = lngamma(`alpha'+x)
		qui: su log_gamma_alpha_p_x
		local log_gamma_alpha_p_x = r(sum)
		drop log_gamma_alpha_p_x
		local posterior = `sum_x'*log(`beta')+`log_gamma_alpha_p_x'-`N'*lngamma(`alpha')-(`alpha'*`N'+`sum_x')*log(1+`beta')
		frame post posterior (`a') (`b') (`alpha') (`beta') (`posterior')
		di "." _continue
	}
}
frame posterior: gen scatter_label="(10.49, 25.2)" if _n==143408
frame posterior: twoway (contourline posterior b a, levels(15)) (scatter b a if _n==143408, mlabel(scatter_label))
